# Projet  — Auto-ML Adaptatif : Analyse Exploratoire des Données (EDA)

**Module :** Stratégies d'exploration des données  
**Enseignant :** Pr. TABBAA Hiba  
**Dataset :** UNSW-NB15 (Network Intrusion Detection)  


---

Ce notebook réalise une analyse exploratoire complète du dataset UNSW-NB15,
suivi d'une simulation du flux de données et d'une visualisation des résultats
du moteur Auto-ML Adaptatif.

In [1]:
# Imports et configuration
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Chemin vers le répertoire racine du projet
RACINE = Path('..').resolve()
sys.path.insert(0, str(RACINE))

DATA_PATH = RACINE / 'data' / 'UNSW_NB15_training-set.csv'
RESULTS_PATH = RACINE / 'results' / 'history.csv'

print(f'Dataset : {DATA_PATH}')
print(f'Dataset disponible : {DATA_PATH.exists()}')

Dataset : C:\Users\user\Desktop\S1\Data Mining\Auto ML\automl_adaptatif\data\UNSW_NB15_training-set.csv
Dataset disponible : True


---
## Section 1 — Chargement et Aperçu des Données

In [2]:
# Chargement du dataset
if not DATA_PATH.exists():
    print('⚠️  Dataset introuvable !')
    print('Téléchargez UNSW_NB15_training-set.csv depuis :')
    print('https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15')
    print('Et placez-le dans le dossier data/')
else:
    df_raw = pd.read_csv(DATA_PATH)
    print(f'Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
    display(df_raw.head())

Dimensions : 82,332 lignes × 45 colonnes


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.0000,udp,-,INT,2,0,496,0,90909.0902,254,0,180363632.0000,0.0000,0,0,0.0110,0.0000,0.0000,0.0000,0,0,0,0,0.0000,0.0000,0.0000,248,0,0,0,2,2,1,1,1,2,0,0,0,1,2,0,Normal,0
1,2,0.0000,udp,-,INT,2,0,1762,0,125000.0003,254,0,881000000.0000,0.0000,0,0,0.0080,0.0000,0.0000,0.0000,0,0,0,0,0.0000,0.0000,0.0000,881,0,0,0,2,2,1,1,1,2,0,0,0,1,2,0,Normal,0
2,3,0.0000,udp,-,INT,2,0,1068,0,200000.0051,254,0,854400000.0000,0.0000,0,0,0.0050,0.0000,0.0000,0.0000,0,0,0,0,0.0000,0.0000,0.0000,534,0,0,0,3,2,1,1,1,3,0,0,0,1,3,0,Normal,0
3,4,0.0000,udp,-,INT,2,0,900,0,166666.6608,254,0,600000000.0000,0.0000,0,0,0.0060,0.0000,0.0000,0.0000,0,0,0,0,0.0000,0.0000,0.0000,450,0,0,0,3,2,2,2,1,3,0,0,0,2,3,0,Normal,0
4,5,0.0000,udp,-,INT,2,0,2126,0,100000.0025,254,0,850400000.0000,0.0000,0,0,0.0100,0.0000,0.0000,0.0000,0,0,0,0,0.0000,0.0000,0.0000,1063,0,0,0,3,2,2,2,1,3,0,0,0,2,3,0,Normal,0


In [3]:
# Types de données
if DATA_PATH.exists():
    print('Types de données par colonne :')
    print(df_raw.dtypes.value_counts())
    print('\nColonnes non-numériques :')
    print(df_raw.select_dtypes(exclude=[np.number]).columns.tolist())
    print('\nValeurs manquantes :')
    manquants = df_raw.isnull().sum()
    print(manquants[manquants > 0] if manquants.any() else 'Aucune valeur manquante')

Types de données par colonne :
int64      30
float64    11
object      4
Name: count, dtype: int64

Colonnes non-numériques :
['proto', 'service', 'state', 'attack_cat']

Valeurs manquantes :
Aucune valeur manquante


---
## Section 2 — Distribution des Classes (Normal vs Attaque)

In [4]:
if DATA_PATH.exists():
    # Distribution des classes
    dist = df_raw['label'].value_counts()
    labels_noms = {0: 'Normal (0)', 1: 'Attaque (1)'}
    
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Distribution des classes', 'Proportion (%)'],
                        specs=[[{"type": "xy"}, {"type": "domain"}]])
    
    # Barres
    fig.add_trace(go.Bar(
        x=[labels_noms[k] for k in dist.index],
        y=dist.values,
        marker_color=['#2196F3', '#FF5722'],
        text=dist.values,
        textposition='outside',
        name='Nombre d\'instances',
    ), row=1, col=1)
    
    # Camembert
    fig.add_trace(go.Pie(
        labels=[labels_noms[k] for k in dist.index],
        values=dist.values,
        marker_colors=['#2196F3', '#FF5722'],
        hole=0.3,
    ), row=1, col=2)
    
    fig.update_layout(
        title='Distribution des classes dans UNSW-NB15',
        height=400,
        showlegend=False,
    )
    fig.show()
    
    print(f'\nTaux de déséquilibre : {dist[0]/dist[1]:.2f}:1 (Normal:Attaque)')

ValueError: Trace type 'pie' is not compatible with subplot type 'xy'
at grid position (1, 2)

See the docstring for the specs argument to plotly.subplots.make_subplots
for more information on subplot types

---
## Section 3 — Statistiques Descriptives

In [ ]:
if DATA_PATH.exists():
    # Supprimer les colonnes non-numériques pour les stats
    colonnes_non_num = ['proto', 'service', 'state', 'attack_cat']
    df_num = df_raw.drop(columns=[c for c in colonnes_non_num if c in df_raw.columns])
    df_num = df_num.select_dtypes(include=[np.number])
    
    print('Statistiques descriptives des features numériques :')
    display(df_num.describe().T.style.background_gradient(cmap='Blues'))

---
## Section 4 — Heatmap des Corrélations

In [ ]:
if DATA_PATH.exists():
    # Calculer les corrélations
    corr = df_num.corr()
    
    # Sélectionner les 20 features les plus corrélées avec le label
    if 'label' in corr:
        top_features = corr['label'].abs().sort_values(ascending=False).head(20).index.tolist()
        corr_top = corr.loc[top_features, top_features]
    else:
        corr_top = corr
    
    fig_corr = px.imshow(
        corr_top,
        title='Heatmap des corrélations entre features numériques (top 20)',
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        height=700,
        text_auto='.2f',
    )
    fig_corr.show()
    
    if 'label' in corr:
        print('\nTop 10 features les plus corrélées avec le label :')
        print(corr['label'].abs().sort_values(ascending=False).head(11).drop('label'))

---
## Section 5 — Distribution des Features les Plus Discriminantes

In [ ]:
if DATA_PATH.exists() and 'label' in df_num.columns:
    # Sélectionner les 10 features les plus discriminantes
    top10 = corr['label'].abs().sort_values(ascending=False).head(11).drop('label').index.tolist()[:10]
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for i, feature in enumerate(top10):
        for classe, couleur, label in [(0, '#2196F3', 'Normal'), (1, '#FF5722', 'Attaque')]:
            data = df_num[df_num['label'] == classe][feature].dropna()
            # Limiter à 95e percentile pour éviter les outliers extrêmes
            p95 = data.quantile(0.95)
            data = data[data <= p95]
            axes[i].hist(data, bins=30, alpha=0.6, color=couleur, label=label, density=True)
        
        axes[i].set_title(f'{feature}', fontsize=10)
        axes[i].set_xlabel('')
        if i == 0:
            axes[i].legend(fontsize=8)
    
    plt.suptitle('Distribution des 10 features les plus discriminantes (Normal vs Attaque)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## Section 6 — Simulation du Flux (1000 premières instances)

In [ ]:
if DATA_PATH.exists():
    from src.stream import stream_dataset
    
    # Collecter les 1000 premières instances du flux
    labels_flux = []
    n_collecte = 1000
    
    for i, (x, y) in enumerate(stream_dataset(str(DATA_PATH), shuffle=False)):
        labels_flux.append(y)
        if i >= n_collecte - 1:
            break
    
    # Visualiser les labels comme un flux temporel
    labels_cumul = pd.Series(labels_flux).cumsum() / (pd.Series(range(1, len(labels_flux)+1)))
    
    fig_flux = make_subplots(rows=2, cols=1,
                             subplot_titles=['Labels bruts (0=Normal, 1=Attaque)',
                                            'Proportion cumulative d\'attaques'])
    
    fig_flux.add_trace(go.Scatter(
        y=labels_flux, mode='markers',
        marker=dict(color=labels_flux, colorscale=['#2196F3', '#FF5722'], size=4),
        name='Label',
    ), row=1, col=1)
    
    fig_flux.add_trace(go.Scatter(
        y=labels_cumul.values, mode='lines',
        line=dict(color='#9C27B0', width=2),
        name='Proportion cumulative',
    ), row=2, col=1)
    
    fig_flux.update_layout(
        title=f'Simulation du flux — {n_collecte} premières instances',
        height=600,
    )
    fig_flux.show()

---
## Section 7 — Test de la Dérive Artificielle

In [ ]:
if DATA_PATH.exists():
    from src.stream import stream_with_artificial_drift
    from src.drift import DriftDetector
    
    DRIFT_AT = 500
    N_TOTAL = 1200
    
    # Simuler le flux avec dérive artificielle
    erreurs = []
    predictions_HT = []
    
    from river import tree
    modele_test = tree.HoeffdingTreeClassifier(grace_period=50)
    detecteur_test = DriftDetector(adwin_delta=0.002)
    
    drifts_detectees = []
    
    for i, (x, y) in enumerate(stream_with_artificial_drift(str(DATA_PATH), drift_at=DRIFT_AT)):
        y_pred = modele_test.predict_one(x)
        if y_pred is None:
            y_pred = 0
        erreur = int(y_pred != y)
        erreurs.append(erreur)
        
        detecteur_test.update(float(erreur))
        if detecteur_test.drift_detected():
            drifts_detectees.append(i)
            detecteur_test.reset()
        
        modele_test.learn_one(x, y)
        
        if i >= N_TOTAL - 1:
            break
    
    # Calcul du taux d'erreur glissant (fenêtre de 50)
    fenetre = 50
    taux_erreur = [sum(erreurs[max(0,i-fenetre):i+1])/(min(i+1, fenetre)) 
                   for i in range(len(erreurs))]
    
    fig_drift = go.Figure()
    
    fig_drift.add_trace(go.Scatter(
        y=taux_erreur,
        mode='lines',
        name='Taux d\'erreur (fenêtre=50)',
        line=dict(color='#2196F3', width=2),
    ))
    
    # Ligne de dérive artificielle
    fig_drift.add_vline(
        x=DRIFT_AT,
        line_dash='dash',
        line_color='orange',
        annotation_text='Dérive artificielle introduite',
    )
    
    # Dérives détectées par ADWIN/PageHinkley
    for d in drifts_detectees:
        fig_drift.add_vline(
            x=d,
            line_dash='dot',
            line_color='red',
            annotation_text=f'Dérive détectée (#{drifts_detectees.index(d)+1})',
        )
    
    fig_drift.update_layout(
        title=f'Test de dérive artificielle — HT + ADWIN+PageHinkley (drift_at={DRIFT_AT})',
        xaxis_title='Numéro d\'instance',
        yaxis_title='Taux d\'erreur glissant',
        height=400,
    )
    fig_drift.show()
    
    print(f'Dérive artificielle introduite à : instance {DRIFT_AT}')
    print(f'Dérives détectées par ADWIN/PH : {drifts_detectees}')

---
## Section 8 — Résultats du Moteur Auto-ML

In [ ]:
if RESULTS_PATH.exists():
    df_results = pd.read_csv(RESULTS_PATH)
    print(f'Historique chargé : {len(df_results)} entrées')
    display(df_results.head())
else:
    print('⚠️  Pas encore de résultats. Lancez le moteur Auto-ML d\'abord :')
    print('    python -m src.automl_engine')

In [ ]:
if RESULTS_PATH.exists() and 'df_results' in dir():
    # Graphique de précision par modèle
    colonnes_acc = [c for c in df_results.columns if c.startswith('accuracy_')]
    
    if colonnes_acc:
        df_plot = df_results[['instance'] + colonnes_acc].copy()
        df_melted = df_plot.melt(id_vars='instance', var_name='Modèle', value_name='Précision')
        df_melted['Modèle'] = df_melted['Modèle'].str.replace('accuracy_', '')
        
        fig_res = px.line(
            df_melted,
            x='instance', y='Précision', color='Modèle',
            title='Précision glissante des modèles — Résultats Auto-ML',
            labels={'instance': 'Numéro d\'instance'},
            color_discrete_map={'HT': '#2196F3', 'KNN': '#FF9800', 'SGD': '#4CAF50'},
            height=450,
        )
        fig_res.show()
    
    # Statistiques finales
    print('\n=== Résultats finaux ===')
    for col in colonnes_acc:
        nom = col.replace('accuracy_', '')
        print(f'{nom}: précision finale = {df_results[col].iloc[-1]:.4f}')
    
    print(f'\nModèle actif final : {df_results["modele_actif"].iloc[-1]}')
    print(f'Dérives détectées : {df_results["nb_drifts"].iloc[-1]}')
    print(f'Instances traitées : {df_results["instance"].iloc[-1]:,}')

In [ ]:
if RESULTS_PATH.exists() and 'df_results' in dir():
    # Graphique de latence
    colonnes_lat = [c for c in df_results.columns if c.startswith('latency_ms_')]
    
    if colonnes_lat:
        df_lat = df_results[['instance'] + colonnes_lat].copy()
        df_lat_melted = df_lat.melt(id_vars='instance', var_name='Modèle', value_name='Latence (ms)')
        df_lat_melted['Modèle'] = df_lat_melted['Modèle'].str.replace('latency_ms_', '')
        
        fig_lat = px.line(
            df_lat_melted,
            x='instance', y='Latence (ms)', color='Modèle',
            title='Latence de prédiction par modèle (ms)',
            color_discrete_map={'HT': '#2196F3', 'KNN': '#FF9800', 'SGD': '#4CAF50'},
            height=350,
        )
        fig_lat.show()
    
    # Distribution du modèle actif
    if 'modele_actif' in df_results.columns:
        dist_modele = df_results['modele_actif'].value_counts()
        fig_dist = px.bar(
            x=dist_modele.index, y=dist_modele.values,
            title='Fréquence de sélection de chaque modèle',
            labels={'x': 'Modèle', 'y': 'Nombre de fois sélectionné'},
            color=dist_modele.index,
            color_discrete_map={'HT': '#2196F3', 'KNN': '#FF9800', 'SGD': '#4CAF50'},
            height=350,
        )
        fig_dist.show()

---
## Conclusion

Ce notebook a permis de :
1. Explorer la structure et la distribution du dataset UNSW-NB15
2. Identifier les features les plus discriminantes pour la détection d'intrusions
3. Simuler un flux de données incrémental
4. Vérifier que le détecteur de dérive (ADWIN + Page-Hinkley) réagit bien aux changements de distribution
5. Visualiser les performances du moteur Auto-ML Adaptatif

**Prochaines étapes :**
- Lancer le moteur complet sur 50 000 instances
- Optimiser les hyperparamètres avec Optuna
- Analyser les résultats dans le rapport scientifique